In [2]:
import os
import json

import requests
import weaviate
import weaviate.classes as wvc
from weaviate.classes.query import MetadataQuery, Filter
from weaviate.embedded import EmbeddedOptions

from getpass import getpass

In [3]:
os.environ['OPENAI_API_KEY'] = getpass('Provide OPEN_API_KEY')

In [4]:
def jprint(value):
    print(json.dumps(value, indent=2, default=str))


def load_tiny_jeopardy():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json"
    return requests.get(url, timeout=30).json()


def load_jeopardy_1k():
    url = "https://raw.githubusercontent.com/weaviate-tutorials/intro-workshop/main/data/jeopardy_1k.json"
    return requests.get(url, timeout=30).json()


def connect_local():
    if "OPENAI_API_KEY" not in os.environ:
        raise RuntimeError("Set OPENAI_API_KEY before running this notebook.")

    headers = {"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]}

    return weaviate.connect_to_local(headers=headers)


def recreate_question_collection(client, *, include_round=False, generative=False):
    if client.collections.exists("Question"):
        client.collections.delete("Question")

    properties = [
        wvc.config.Property(name="question", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="answer", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="category", data_type=wvc.config.DataType.TEXT),
    ]
    if include_round:
        properties.append(wvc.config.Property(name="round", data_type=wvc.config.DataType.TEXT))

    kwargs = {}
    if generative:
        kwargs["generative_config"] = wvc.config.Configure.Generative.openai(model="gpt-4o-mini")

    return client.collections.create(
        name="Question",
        vector_config=wvc.config.Configure.Vectors.text2vec_openai(model="text-embedding-3-small"),
        properties=properties,
        **kwargs,
    )


def import_questions(collection, data, *, include_round=False):
    objects = []
    for d in data:
        obj = {
            "question": d["Question"],
            "answer": d["Answer"],
            "category": d["Category"],
        }
        if include_round:
            obj["round"] = d.get("Round", "")
        objects.append(obj)

    result = collection.data.insert_many(objects)
    if result.errors:
        print("Import errors:")
        jprint(result.errors)
    return result

In [5]:
client = connect_local()

In [6]:
questions = recreate_question_collection(client)

In [7]:
uuid = questions.data.insert(
    {
        'question': 'Leonardo da Vinci was born in this country.',
        'answer': 'Italy',
        'category': 'Culture'
    }
)

print(uuid)

8c8a9da4-0ab0-4ba4-94bb-f0200a89a542


In [8]:
obj = questions.query.fetch_object_by_id(uuid=uuid)
jprint(obj.properties)

{
  "answer": "Italy",
  "question": "Leonardo da Vinci was born in this country.",
  "category": "Culture"
}


In [9]:
obj = questions.query.fetch_object_by_id(uuid=uuid, include_vector=True)
jprint(obj.properties)

{
  "answer": "Italy",
  "category": "Culture",
  "question": "Leonardo da Vinci was born in this country."
}


In [10]:
print('vector length:', len(obj.vector['default']))

vector length: 1536


In [11]:
questions.data.update(
    uuid=uuid,
    properties={'answer': 'Florence, Italy'}
)

obj = questions.query.fetch_object_by_id(uuid=uuid, include_vector=True)
jprint(obj.properties)

{
  "answer": "Florence, Italy",
  "category": "Culture",
  "question": "Leonardo da Vinci was born in this country."
}


In [12]:
questions.data.delete_by_id(uuid)
questions.aggregate.over_all(total_count=True).total_count

0

In [13]:
client.close()